In [ ]:
%load_ext autoreload
%autoreload 2

import mujoco
import numpy as np
import PIL.Image

from swarmbots.unit import Unit
from rendering import display_video

In [ ]:
RENDER_WIDTH = 640
RENDER_HEIGHT = 480

In [ ]:
# Initialize Unit
unit = Unit(
    body_radius=0.1,
    leg_length=0.2,
    leg_radius=0.025,
    hinge_range=np.pi / 8
)

# Access the spec to add environment details
spec = unit.spec
worldbody = spec.worldbody

# Add a freejoint to the main body so it can move
# Note: unit.body is the MjSpecBody for the main body
unit.body.add_joint(type=mujoco.mjtJoint.mjJNT_FREE)

# Add a textured floor
worldbody.add_geom(
    type=mujoco.mjtGeom.mjGEOM_PLANE,
    size=[2, 2, 0.1],
    rgba=[0.2, 0.3, 0.4, 1],
    pos=[0, 0, 0]
)

# Add lighting
worldbody.add_light(pos=[0, 0, 3], dir=[0, 0, -1])
worldbody.add_light(pos=[2, 2, 3], dir=[-1, -1, -1])

# Compile to MjModel
model = unit.to_model()
data = mujoco.MjData(model)
renderer = mujoco.Renderer(model, height=RENDER_HEIGHT, width=RENDER_WIDTH)

print(f"Model compiled. nq={model.nq}, nv={model.nv}")

In [ ]:
# Simulation loop
duration = 5.0  # seconds
framerate = 30  # Hz
frames = []

mujoco.mj_resetData(model, data)

# Set initial position (x, y, z) slightly above ground
# The freejoint adds 7 qpos: x, y, z, qw, qx, qy, qz
data.qpos[0:3] = [0, 0, 0.5]

while data.time < duration:
    mujoco.mj_step(model, data)
    
    if len(frames) < data.time * framerate:
        renderer.update_scene(data)
        frames.append(renderer.render().copy())

print(f"Simulated {len(frames)} frames")
display_video(frames, framerate)